<a href="https://colab.research.google.com/github/zbh3un/4002_proj1/blob/main/official_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install vaderSentiment transformers torch emoji -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

reviews_df = pd.read_csv("reviews.csv")
reviews_df = reviews_df.drop(['Time_submitted', 'Reply', 'Total_thumbsup'], axis=1)
reviews_df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 23.4 MB/s eta 0:00:00


In [ ]:
# clean emojis
reviews_df['Review_cleaned'] = reviews_df['Review'].apply(
    lambda x: emoji.replace_emoji(str(x), '') if pd.notna(x) else ""
)
reviews_df

In [ ]:
# VADER
vader = SentimentIntensityAnalyzer()
compounds = [vader.polarity_scores(t)['compound'] for t in reviews_df['Review_cleaned']]
reviews_df['vader_compound'] = compounds
reviews_df['vader_predicted_rating'] = [int(round((c + 1) * 2 + 1)) for c in compounds]
reviews_df

In [ ]:
# HuggingFace BERT
hf_pipe = pipeline("sentiment-analysis",
                   model="nlptown/bert-base-multilingual-uncased-sentiment",
                   device=0, truncation=True, max_length=512)

texts = [t if t.strip() else "neutral" for t in reviews_df['Review_cleaned']]

results = []
for i in range(0, len(texts), 32):
    results.extend(hf_pipe(texts[i:i+32]))

reviews_df['hf_predicted_rating'] = [int(r['label'].split()[0]) for r in results]
reviews_df

In [ ]:
# accuracy results
vader_exact = (reviews_df['Rating'] == reviews_df['vader_predicted_rating']).mean() * 100
vader_off1 = (abs(reviews_df['Rating'] - reviews_df['vader_predicted_rating']) <= 1).mean() * 100
vader_diff = abs(reviews_df['Rating'] - reviews_df['vader_predicted_rating']).mean()

hf_exact = (reviews_df['Rating'] == reviews_df['hf_predicted_rating']).mean() * 100
hf_off1 = (abs(reviews_df['Rating'] - reviews_df['hf_predicted_rating']) <= 1).mean() * 100
hf_diff = abs(reviews_df['Rating'] - reviews_df['hf_predicted_rating']).mean()

pd.DataFrame({
    'Model': ['VADER', 'HuggingFace'],
    'Exact Match %': [vader_exact, hf_exact],
    'Off-by-One %': [vader_off1, hf_off1],
    'Avg Error': [vader_diff, hf_diff]
})

In [ ]:
# visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# accuracy comparison
x = np.arange(2)
axes[0, 0].bar(x - 0.175, [vader_exact, hf_exact], 0.35, label='Exact Match', color='steelblue')
axes[0, 0].bar(x + 0.175, [vader_off1, hf_off1], 0.35, label='Off-by-One', color='lightsteelblue')
axes[0, 0].set_ylabel('Accuracy (%)')
axes[0, 0].set_title('Model Accuracy')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(['VADER', 'HuggingFace'])
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 100)

# confusion matrices
sns.heatmap(pd.crosstab(reviews_df['Rating'], reviews_df['vader_predicted_rating']),
            annot=True, fmt='d', cmap='Blues', ax=axes[0, 1])
axes[0, 1].set_title('VADER')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Actual')

sns.heatmap(pd.crosstab(reviews_df['Rating'], reviews_df['hf_predicted_rating']),
            annot=True, fmt='d', cmap='Greens', ax=axes[0, 2])
axes[0, 2].set_title('HuggingFace')
axes[0, 2].set_xlabel('Predicted')
axes[0, 2].set_ylabel('Actual')

# error distributions
axes[1, 0].hist(reviews_df['Rating'] - reviews_df['vader_predicted_rating'],
                bins=9, range=(-4.5, 4.5), color='steelblue', edgecolor='black')
axes[1, 0].set_title('VADER Errors')
axes[1, 0].axvline(0, color='red', linestyle='--')

axes[1, 1].hist(reviews_df['Rating'] - reviews_df['hf_predicted_rating'],
                bins=9, range=(-4.5, 4.5), color='green', edgecolor='black')
axes[1, 1].set_title('HuggingFace Errors')
axes[1, 1].axvline(0, color='red', linestyle='--')

# rating distribution
x = np.arange(1, 6)
actual = [sum(reviews_df['Rating'] == i) for i in x]
vader_pred = [sum(reviews_df['vader_predicted_rating'] == i) for i in x]
hf_pred = [sum(reviews_df['hf_predicted_rating'] == i) for i in x]
axes[1, 2].bar(x - 0.25, actual, 0.25, label='Actual', color='gray')
axes[1, 2].bar(x, vader_pred, 0.25, label='VADER', color='steelblue')
axes[1, 2].bar(x + 0.25, hf_pred, 0.25, label='HuggingFace', color='green')
axes[1, 2].set_title('Rating Distribution')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# examples where HuggingFace is better
sample_df = reviews_df[['Review', 'Rating', 'vader_predicted_rating', 'hf_predicted_rating']].copy()
sample_df['vader_error'] = abs(sample_df['Rating'] - sample_df['vader_predicted_rating'])
sample_df['hf_error'] = abs(sample_df['Rating'] - sample_df['hf_predicted_rating'])

sample_df[sample_df['hf_error'] < sample_df['vader_error']].head(5)

In [ ]:
# examples where VADER is better
sample_df[sample_df['vader_error'] < sample_df['hf_error']].head(5)